# <font style="color:blue">Assignment: Implement the Focal Loss</font>

We are already familiar with detection loss function; this is a combination of location loss and classification loss. Remember that we have used smooth L1 loss for location loss and OHEM loss for classification loss. 

In this assignment, you have to implement the **focal loss** for classification loss. Location loss will remain as it is.

## <font color='blue'>Marking Scheme</font>

#### Maximum Points: 30

<div>
    <table>
        <tr><td><h3>Sr. no.</h3></td> <td><h3>Problem</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1</h3></td> <td><h3>2. Focal loss Implementation</h3></td> <td><h3>30</h3></td> </tr>
    </table>
</div>

# <font style="color:green">1. Focal Loss</font>

**Following is the screenshot form the RetinaNet lecture. It has the definition of focal loss.**

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w9-focal_loss.png'>

<p></p>

Originally this is published in the paper [Focal Loss for Dense Object Detection](https://arxiv.org/pdf/1708.02002.pdf).

# <font style="color:green">2. Focal loss Implementation [30 Points]</font>

We have defined the DetectionLoss class, where smooth L1 loss is already implemented. You do not have to make any changes in this part. **You have to implement the focal loss part**. 

Keep in mind that class targets with label `-1` must be ignored at the time of calculating the **focal loss**. The value of gamma we have chosen `2`; do not change it. 

**Hints:** 

- The following link may be useful to understand a few loss function implementation in PyTorch. Understanding those will be very helpful in the focal loss implementation.


- https://pytorch.org/docs/stable/nn.html#torch.nn.CrossEntropyLoss


- https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.cross_entropy


- https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.log_softmax


- https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.nll_loss


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math

**Write your code where it is specified. Do not modify / delete other codes.**

In [27]:
import numpy as np 
class DetectionLoss(nn.Module):
    def __init__(self, num_classes, gamma=2, ignore_index=-1):
        super().__init__()
        self.num_classes = num_classes
        
        # gamma will be used in focal loss
        self.gamma = gamma
        
        # ignore_index will be used for classification loss.
        # at the time of encoding, anchor boxes with 0.4 <IoU < 0.5, assign -1 as label.
        # at the time of finding the classification loss, these labels should be ignored
        self.ignore_index = ignore_index

    def forward(self, loc_preds, loc_targets, cls_preds, cls_targets):
        '''Compute loss between (loc_preds, loc_targets) and (cls_preds, cls_targets).

        Args:
          loc_preds: (tensor) predicted locations, sized [batch_size, #anchors, 4].
          loc_targets: (tensor) encoded target locations, sized [batch_size, #anchors, 4].
          cls_preds: (tensor) predicted class confidences, sized [batch_size, #anchors, #classes].
          cls_targets: (tensor) encoded target labels, sized [batch_size, #anchors].

        loss:
          (tensor) loss = (SmoothL1Loss(loc_preds, loc_targets),  FocalLoss(cls_preds, cls_targets)).
        '''
        loc_loss = 0.0
        cls_loss = 0.0
        ################################################################
        # loc_loss
        ################################################################
        # print(f" loc_preds shape: {loc_preds.shape}     loc_targets: {loc_targets.shape}")
        # print(f" cls_preds shape: {cls_preds.shape}     cls_targets: {cls_targets.shape}")
        # print(f" cls_targets    : {cls_targets[0,:5]}   {cls_targets.long().sum()}")
    
        pos = cls_targets > 0  # [N,#anchors]
        # _tmp = np.nonzero(cls_targets)
        # print(_tmp.shape, _tmp)
        
        # print(cls_targets[0,_tmp[:,1]])
        
        num_pos = pos.long().sum(1, keepdim=True)
        # print(f" pos: {pos.shape}  -  num pos : {num_pos} ")
        # print(f" pos: {pos.unsqueeze(2).shape}    pos: {pos.unsqueeze(2).expand_as(loc_preds).shape} ")
        mask = pos.unsqueeze(2).expand_as(loc_preds)  # [N,#anchors,4]
        # print(f" mask:  {mask.shape}")

        
        masked_loc_preds = loc_preds[mask]  # [#pos,4]
        # print(f" 1.masked_loc_preds   :   {masked_loc_preds.shape} ")
        masked_loc_preds = loc_preds[mask].view(-1, 4)  # [#pos,4]
        # print(f" 2.masked_loc_preds   :   {masked_loc_preds.shape} ")
        
        masked_loc_targets = loc_targets[mask]  # [#pos,4]
        # print(f" 3.masked_loc_targets :   {masked_loc_targets.shape}")
        masked_loc_targets = loc_targets[mask].view(-1, 4)  # [#pos,4]
        # print(f" 4.masked_loc_targets :   {masked_loc_targets.shape}")
        
        loc_loss = F.smooth_l1_loss(masked_loc_preds, masked_loc_targets, reduction='none')
        loc_loss = loc_loss.sum() / num_pos.sum().float()

        ################################################################
        # cls_loss with Focal Loss
        ################################################################
        # print("\n\n----------------------------------------------------"
        #       "\n Focal Loss \n"              
        #       "----------------------------------------------------\n\n")    
        # inv = (cls_targets == -1)
        # bg  = (cls_targets == 0)
        # obj = (cls_targets > 0)
        pos = (cls_targets >= 0)
        mask = pos.unsqueeze(2).expand_as(cls_preds)

        # print(f" # (cls_targets)         : {cls_targets.shape}") 
        # print(f" # (cls_targets == -1)   : {inv.sum()}") 
        # print(f" # (cls_targets == 0)    : {bg.sum()}") 
        # print(f" # (cls_targets >  0)    : {obj.sum()}") 
        # print(f" # (cls_targets >= 0 )   : {pos.sum()}") 
        # print(f" mask                    : {mask.shape}")
        class_count = cls_preds.shape[-1]
        print(f" cls_preds.shape         : {cls_preds.shape}    {class_count}  dtype:  {cls_preds.dtype} ")
        print(f" cls_targets.shape       : {cls_targets.shape}    dtype:  {cls_targets.dtype}")
        
        masked_cls_preds = cls_preds[mask].view(-1, class_count)  # [#pos,4]
        masked_cls_probs = F.softmax(masked_cls_preds, dim = -1)
        masked_cls_targets = cls_targets[mask[...,0]].view(-1,1)
        masked_cls_targets_bool = (masked_cls_targets > 0).long().squeeze()
        
        # print(f" masked_cls_preds        : {masked_cls_preds.shape}   dtype: { masked_cls_preds.dtype}")
        # print(f" masked_cls_probs        : {masked_cls_probs.shape}   dtype: { masked_cls_probs.dtype}" )
        # print(f" masked_cls_targets      : {masked_cls_targets.shape}   dtype: {masked_cls_targets.dtype}")
        # print(f" masked_cls_targets_bool : {masked_cls_targets_bool.shape}   dtype: {masked_cls_targets_bool.dtype}")
        
        # test_rows = np.arange(16900, 16911)
        # for i in test_rows:
        #     print(f" \t row: {i}   masked_cls_targets:  {masked_cls_targets[i].item()}     masked_cls_targets_bool: {masked_cls_targets_bool[i]}")        
        #
        # print(f" {masked_cls_preds[0:2]} ")
        # print(f" {masked_cls_probs[0:2]} ")
        # print(f" {masked_cls_probs[0:2].sum(dim=1)} ")
        # print(f"  masked_cls_targets : {masked_cls_targets.shape} ")

        p = torch.gather(masked_cls_probs, 1, masked_cls_targets).squeeze() 
        pt  = (p*masked_cls_targets_bool) + (1-p)*(1-masked_cls_targets_bool)
        

        # test_rows = np.arange(16900, 16911)
        # for i in test_rows:
        #     print(f" \t pred_softmax : {masked_cls_probs[i]}    tgt: {masked_cls_targets_bool[i].item():2d}  tgt_cls:  {masked_cls_targets[i].item():2d}"
        #           f"   p[i]:  {p[i].item():.5f}   pt[i]: {pt[i].item():.5f}")        ###
        #
        # print(f" masked_cls_targets      : {masked_cls_targets.shape}  dtype: {masked_cls_targets.dtype}")
        # print(f" masked_cls_preds        : {masked_cls_preds.shape}  dtype:  {masked_cls_preds.dtype}")
        
        CE_loss = F.cross_entropy(masked_cls_preds, masked_cls_targets.squeeze(), ignore_index=self.ignore_index, reduction='none')
        print(f" CE_loss                 : {CE_loss.shape}  dtype:  {CE_loss.dtype}")
        pt_2 = np.exp(-CE_loss)
        print(f" p.shape: {p.shape}    pt.shape: {pt.shape}    pt2.shape: {pt_2.shape}")
        test_rows = np.arange(16900, 16911)        
        for i in test_rows:
            print(f" \t pred_softmax : {masked_cls_probs[i]}    tgt: {masked_cls_targets_bool[i].item():2d}  tgt_cls:  {masked_cls_targets[i].item():2d}"
                  f"   p[i]:  {p[i].item():.5f}   pt[i]: {pt[i].item():.8f}   pt_2: {pt_2[i].item():.8f}"
                  f" CEloss: {CE_loss[i].item():.5f} "
                 )       
        
        nonzero_rows = np.nonzero((masked_cls_targets > 0))[:,0]
        nonzero_count = nonzero_rows.shape[0]
        background_rows  = np.nonzero((masked_cls_targets == 0))[:,0]
        background_count = background_rows.shape[0] 
        background_count2 = (masked_cls_targets == 0).int().sum()
        # print(f" nonzero_rows            :  {nonzero_count}   {nonzero_rows.shape}")
        # print(f" nonzero_rows            :  {nonzero_rows[:10]}")
        print(f" background_rows         :  {background_count}   {background_rows.shape}   {background_count2}")
        # print(f" background_rows         :  {background_rows[:10]}")
        # print(f" pt^gamma shape          :  {(pt**gamma).shape}    CE_loss shape:  {CE_loss.shape}")

        focal_loss = (pt ** gamma) * CE_loss
        
        # print(f" focal loss shape        : {focal_loss.shape}  dtype: {focal_loss.dtype}")
        # print(f" focal loss - objects    : {focal_loss[nonzero_rows]}     mean: {focal_loss[nonzero_rows].mean():.6f}")
        print(f" focal loss - backgorund : {focal_loss[background_rows]}  mean: {focal_loss[background_rows].mean():.6f}")
        print(f" focal loss sum() / 17439: {focal_loss.mean()}" )
        print(f" focal loss.sum() / 17433: {focal_loss.sum()/17433.0}")
        # test_rows = np.arange(16900, 16911)        
        # for i in test_rows:
        #     print(f" \t pred_softmax : {masked_cls_probs[i]}    tgt: {masked_cls_targets_bool[i].item():2d}  tgt_cls:  {masked_cls_targets[i].item():2d}"
        #           f"   p[i]:  {p[i].item():.5f}   pt[i]: {pt[i].item():.8f}"
        #           f" CEloss: {CE_loss[i].item():.5f} "
        #          )
        cls_loss = focal_loss.sum() / background_count
        return loc_loss, cls_loss

In [28]:
img_height = img_width = 300
num_classes = 5  # including background

bounding_boxes = torch.tensor([[100, 100, 150, 150], [120, 200, 160, 250]], dtype=torch.float)
targets = torch.tensor([2, 4])

data_encoder = DataEncoder((img_height, img_width))

num_anchors = data_encoder.get_num_anchors()

bboxes, labels = data_encoder.encode(bounding_boxes, targets)
# print(f" bboxes: {bboxes.shape}    labels : {labels.shape}")
torch.manual_seed(21)

pred_bboxes = torch.rand((1, labels.size()[0], 4))
pred = torch.rand((1, labels.size()[0], num_classes))

# _tmp = np.nonzero(labels)
# print(_tmp)

# for a,b in zip(pred[0,:10], labels[:10]):
#     print(f" {a}   - {b}")

# for a in _tmp:
#     print(f" {pred[0,a,:]}   - {labels[a]}")

    
gamma = 2
detection_loss = DetectionLoss(num_classes, gamma)

bb_loss, cls_pred_loss = detection_loss(pred_bboxes, bboxes.unsqueeze(0), pred, labels.unsqueeze(0))
print('\n\n')
print('Bounding Box Loss:\t{0:.5}'.format(bb_loss))
print('Classification Loss:\t{0:.5}'.format(cls_pred_loss))

 cls_preds.shape         : torch.Size([1, 17451, 5])    5  dtype:  torch.float32 
 cls_targets.shape       : torch.Size([1, 17451])    dtype:  torch.int64
 CE_loss                 : torch.Size([17439])  dtype:  torch.float32
 p.shape: torch.Size([17439])    pt.shape: torch.Size([17439])    pt2.shape: torch.Size([17439])
 	 pred_softmax : tensor([0.1621, 0.1684, 0.2897, 0.1550, 0.2248])    tgt:  0  tgt_cls:   0   p[i]:  0.16214   pt[i]: 0.83786196   pt_2: 0.16213806 CEloss: 1.81931 
 	 pred_softmax : tensor([0.2193, 0.2769, 0.1426, 0.1869, 0.1743])    tgt:  1  tgt_cls:   4   p[i]:  0.17429   pt[i]: 0.17429096   pt_2: 0.17429094 CEloss: 1.74703 
 	 pred_softmax : tensor([0.2545, 0.2440, 0.1304, 0.1768, 0.1943])    tgt:  1  tgt_cls:   4   p[i]:  0.19435   pt[i]: 0.19434787   pt_2: 0.19434786 CEloss: 1.63811 
 	 pred_softmax : tensor([0.1373, 0.1552, 0.2041, 0.2582, 0.2452])    tgt:  0  tgt_cls:   0   p[i]:  0.13732   pt[i]: 0.86268198   pt_2: 0.13731803 CEloss: 1.98546 
 	 pred_softmax : 

/tmp/ipykernel_182326/1226971667.py:117: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  pt_2 = np.exp(-CE_loss)


# <font style="color:green">3. Check the implementation</font>

**Before submitting the notebook, make sure you have verified your implementation.**

Let's write a data encoder class to generate location labels and class labels. We are already familiar with this class in [Create Custom Single-stage Detector](https://courses.opencv.org/courses/course-v1:OpenCV+OpenCV-106+2019_T1/courseware/2ae52496773c42ba8216cca380ad4fd3/2c916b45595d459c8c7b944038512ba9/4?activate_block_id=block-v1%3AOpenCV%2BOpenCV-106%2B2019_T1%2Btype%40vertical%2Bblock%40400083ecaadf4cc392bfd643d899fd5c) section. 

In [5]:
class DataEncoder:
    def __init__(self, input_size):
        self.input_size = input_size
        self.anchor_areas = [8 * 8, 16 * 16., 32 * 32., 64 * 64., 128 * 128]  # p3 -> p7
        self.aspect_ratios = [0.5, 1, 2]
        self.scales = [1, pow(2, 1 / 3.), pow(2, 2 / 3.)]
        num_fms = len(self.anchor_areas)
        fm_sizes = [math.ceil(self.input_size[0] / pow(2., i + 3)) for i in range(num_fms)]
        self.anchor_boxes = []
        for i, fm_size in enumerate(fm_sizes):
            anchors = self.generate_anchors(self.anchor_areas[i], self.aspect_ratios, self.scales)
            anchor_grid = self.generate_anchor_grid(input_size, fm_size, anchors)
            self.anchor_boxes.append(anchor_grid)
        self.anchor_boxes = torch.cat(self.anchor_boxes, 0)

    def encode(self, boxes, classes):
        iou = self.compute_iou(boxes, self.anchor_boxes)
        iou, ids = iou.max(1)
        loc_targets = self.encode_boxes(boxes[ids], self.anchor_boxes)
        cls_targets = classes[ids]
        cls_targets[iou < 0.5] = -1
        cls_targets[iou < 0.4] = 0

        return loc_targets, cls_targets
    
    def get_num_anchors(self):
        return len(self.aspect_ratios) * len(self.scales)
    
    @staticmethod
    def encode_boxes(boxes, anchors):
        anchors_wh = anchors[:, 2:] - anchors[:, :2] + 1
        anchors_ctr = anchors[:, :2] + 0.5 * anchors_wh
        boxes_wh = boxes[:, 2:] - boxes[:, :2] + 1
        boxes_ctr = boxes[:, :2] + 0.5 * boxes_wh
        return torch.cat([(boxes_ctr - anchors_ctr) / anchors_wh, torch.log(boxes_wh / anchors_wh)], 1)
    
    @staticmethod
    def generate_anchor_grid(input_size, fm_size, anchors):
        grid_size = input_size[0] / fm_size
        x, y = torch.meshgrid(torch.arange(0, fm_size) * grid_size, torch.arange(0, fm_size) * grid_size)
        anchors = anchors.view(-1, 1, 1, 4)
        xyxy = torch.stack([x, y, x, y], 2).float()
        boxes = (xyxy + anchors).permute(2, 1, 0, 3).contiguous().view(-1, 4)
        boxes[:, 0::2] = boxes[:, 0::2].clamp(0, input_size[0])
        boxes[:, 1::2] = boxes[:, 1::2].clamp(0, input_size[1])
        return boxes
    
    @staticmethod
    def generate_anchors(anchor_area, aspect_ratios, scales):
        anchors = []
        for scale in scales:
            for ratio in aspect_ratios:
                h = round(math.sqrt(anchor_area) / ratio)
                w = round(ratio * h)
                x1 = (math.sqrt(anchor_area) - scale * w) * 0.5
                y1 = (math.sqrt(anchor_area) - scale * h) * 0.5
                x2 = (math.sqrt(anchor_area) + scale * w) * 0.5
                y2 = (math.sqrt(anchor_area) + scale * h) * 0.5
                anchors.append([x1, y1, x2, y2])
        return torch.Tensor(anchors)
    
    @staticmethod
    def compute_iou(src, dst):
        p1 = torch.max(dst[:, None, :2], src[:, :2])
        p2 = torch.min(dst[:, None, 2:], src[:, 2:])
        inter = torch.prod((p2 - p1 + 1).clamp(0), 2)
        src_area = torch.prod(src[:, 2:] - src[:, :2] + 1, 1)
        dst_area = torch.prod(dst[:, 2:] - dst[:, :2] + 1, 1)
        iou = inter / (dst_area[:, None] + src_area - inter)
        return iou


**Running the below cell, you should get the following outputs:**

```
Bounding Box Loss:	  0.76351
Classification Loss:	1.0741
```

**Bounding box loss must match as its codes are all ready given.**

**If the classification loss does not match but close to it; then there are the possibilities of the following:**


- **If the loss is greater than expected loss:** You might not have taken care of the ignore index.


- **If the loss is less than expected loss:** You might have taken care of the ignore index at the time of calculating the loss but might be forgotten at the time of taking the mean.

In [133]:

img_height = img_width = 300

num_classes = 5  # including background

bounding_boxes = torch.tensor([[100, 100, 150, 150], [120, 200, 160, 250]], dtype=torch.float)
targets = torch.tensor([2, 4])

data_encoder = DataEncoder((img_height, img_width))

num_anchors = data_encoder.get_num_anchors()

bboxes, labels = data_encoder.encode(bounding_boxes, targets)
print(f" bboxes: {bboxes.shape}    labels : {labels.shape}")
torch.manual_seed(21)

pred_bboxes = torch.rand((1, labels.size()[0], 4))
pred = torch.rand((1, labels.size()[0], num_classes))

_tmp = np.nonzero(labels)
print(_tmp)

for a,b in zip(pred[0,:10], labels[:10]):
    print(f" {a}   - {b}")

for a in _tmp:
    print(f" {pred[0,a,:]}   - {labels[a]}")

    
gamma = 2
detection_loss = DetectionLoss(num_classes, gamma)

bb_loss, cls_pred_loss = detection_loss(pred_bboxes, bboxes.unsqueeze(0), pred, labels.unsqueeze(0))
print('\n\n')
print('Bounding Box Loss:\t{0:.5}'.format(bb_loss))
print('Classification Loss:\t{0:.5}'.format(cls_pred_loss))

 bboxes: torch.Size([17451, 4])    labels : torch.Size([17451])
tensor([[14262],
        [14271],
        [15297],
        [15465],
        [15468],
        [15474],
        [15477],
        [16641],
        [16644],
        [16645],
        [16648],
        [16911],
        [16912],
        [16914],
        [16915],
        [16918],
        [16919],
        [17299]])
 tensor([0.3072, 0.3869, 0.4762, 0.4573, 0.7917])   - 0
 tensor([0.6109, 0.0412, 0.6266, 0.3561, 0.4125])   - 0
 tensor([0.9238, 0.7932, 0.8987, 0.0868, 0.0054])   - 0
 tensor([0.4875, 0.6361, 0.8383, 0.5728, 0.8570])   - 0
 tensor([0.9830, 0.6226, 0.2805, 0.2198, 0.5156])   - 0
 tensor([0.1823, 0.7559, 0.8814, 0.3066, 0.5688])   - 0
 tensor([0.1636, 0.9883, 0.7003, 0.1498, 0.3075])   - 0
 tensor([0.8717, 0.6246, 0.0045, 0.5654, 0.8087])   - 0
 tensor([0.8635, 0.4656, 0.5049, 0.2962, 0.3429])   - 0
 tensor([0.9320, 0.0325, 0.2637, 0.4346, 0.9675])   - 0
 tensor([[0.2292, 0.2933, 0.2106, 0.6374, 0.9637]])   - tensor([-1])


In [134]:

img_height = img_width = 300

num_classes = 5  # including background

bounding_boxes = torch.tensor([[100, 100, 150, 150], [120, 200, 160, 250]], dtype=torch.float)
targets = torch.tensor([2, 4])

data_encoder = DataEncoder((img_height, img_width))

num_anchors = data_encoder.get_num_anchors()

bboxes, labels = data_encoder.encode(bounding_boxes, targets)

torch.manual_seed(21)

pred_bboxes = torch.rand((1, labels.size()[0], 4))
pred = torch.rand((1, labels.size()[0], num_classes))

gamma = 2
detection_loss = DetectionLoss(num_classes, gamma)

bb_loss, cls_pred_loss = detection_loss(pred_bboxes, bboxes.unsqueeze(0), pred, labels.unsqueeze(0))

print('Bounding Box Loss:\t{0:.5}'.format(bb_loss))
print('Classification Loss:\t{0:.5}'.format(cls_pred_loss))



----------------------------------------------------
 Focal Loss 
----------------------------------------------------


 mask                    : torch.Size([1, 17451, 5])
 cls_preds.shape         : torch.Size([1, 17451, 5])   mask:  torch.Size([1, 17451, 5]) 
 cls_targets.shape       : torch.Size([1, 17451])    mask:  torch.Size([1, 17451, 5])
 masked_cls_preds        : torch.Size([17439, 5]) 
 masked_cls_preds_softmax: torch.Size([17439, 5])
 masked_cls_targets      : torch.Size([17439, 1])  torch.int64
 masked_cls_targets_bool : torch.Size([17439])  torch.int64
 masked_cls_targets      : torch.Size([17439, 1])  dtype: torch.int64
 masked_cls_preds        : torch.Size([17439, 5])  dtype:  torch.float32
 CE_loss                 :  torch.Size([17439])    torch.float32
 nonzero_rows   :  6   torch.Size([6])
 tensor([15464, 16901, 16902, 16904, 16905, 16908])
 background_rows :   17433   torch.Size([17433])
 tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
torch.Size([17439]) torch.Size([17439

In [116]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###
